# APS360 Handwriting Recognition: Step 1 (IAM data pipeline)

This notebook prepares the IAM word images for the CRNN + CTC model. It:
1. mounts your Google Drive and writes the project code into the session;
2. unpacks `archive.zip` (IAM words) and finds `words.txt` and the images, whatever layout the download uses;
3. **recovers the labels missing from the cut-off Kaggle `words.txt`**, once, saving `words_recovered.txt` to your project folder;
4. runs the tests, then filters and splits IAM, prints the numbers for the report, and shows a sample batch.

**Run:** Runtime → Run all. CPU is fine for this step. When Colab asks for Drive access, allow it.
Your Drive files are never changed; the only things added are `outputs/` and `words_recovered.txt`.

## Mount Google Drive
Your Drive appears at `/content/drive/MyDrive`. Allow access when Colab asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paths
`PROJECT_DIR` is the Drive folder that holds `archive.zip`. If it has been moved, the cell searches your Drive for it.
Heavy files go on Colab's local disk (`/content`), which is far faster than Drive. Results are saved back to Drive.

In [ ]:
import os, subprocess
PROJECT_DIR = "/content/drive/MyDrive/University (Mech Eng)/Year 3/Fall/APS360 (Introduction to Machine Learning)/Project/Project Progress Report"
if not os.path.exists(os.path.join(PROJECT_DIR, "archive.zip")):
    print("archive.zip not found at PROJECT_DIR; searching Drive (can take a minute)...")
    hits = [h for h in subprocess.run(["find", "/content/drive/MyDrive", "-name", "archive.zip", "-size", "+500M"],
                                      capture_output=True, text=True).stdout.split("\n") if h]
    assert hits, "Could not find archive.zip anywhere in your Drive"
    PROJECT_DIR = os.path.dirname(hits[0])

ARCHIVE    = os.path.join(PROJECT_DIR, "archive.zip")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")       # results saved back to Drive
REPORT_DIR = os.path.join(OUTPUT_DIR, "report")         # figures/tables for the progress report
OFFICIAL_WORDS_TXT = os.path.join(PROJECT_DIR, "words.txt")            # full official file, if you add it
RECOVERED_WORDS_TXT = os.path.join(PROJECT_DIR, "words_recovered.txt") # made by the label-recovery step
CODE_DIR   = "/content/aps360_htr"                      # project code (this session only)
RAW_DIR    = "/content/iam_raw"                         # archive.zip unpacked here
DATA_ROOT  = "/content/iam"                             # tidy IAM layout (symlinks into RAW_DIR)
os.environ["HTR_DATA_ROOT"] = DATA_ROOT                 # src/config.py reads this
os.makedirs(REPORT_DIR, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)

## Write the project code
The data code is shown below as `%%writefile` cells you can read. Model and training code (Step 2) comes from the embedded zip.

| File | What it does |
|---|---|
| `src/config.py` | all settings: data path, image height 32, width limits, filters, model and training defaults |
| `src/data/charset.py` | characters ↔ numbers; CTC blank = 0; greedy CTC decoding |
| `src/data/transforms.py` | resize to 32 px high (keeping aspect ratio), normalize to [-1, 1] |
| `src/data/iam.py` | read `words.txt`, filter bad samples, writer-disjoint splits, `Dataset`, `collate_fn` |
| `src/utils/metrics.py` | CER and WER |
| `src/data/recover.py`, `scripts/recover_labels.py` | rebuild missing `words.txt` lines from IAM's embedded XML |
| `scripts/setup_iam.py` | find `words.txt` and the images in any download layout |
| `scripts/inspect_data.py` | statistics + a picture of one batch |
| `data/` | Aachen split lists + the form → writer table (`iam_form_writers.csv`, see its `_SOURCE.md`) |

### The rest of the codebase (embedded zip)
Every step notebook is generated from one repo; this one embeds code **v16** (`aps360_htr_code_v16.zip` in your project folder).
This cell unpacks the 71 files that aren't shown below (see the other notebook or the zip to read them).
The long string is just that zip, base64-encoded.

In [ ]:
import base64, io, os, shutil, sys, zipfile
# Start from a clean copy every time: delete any older code left in /content/aps360_htr,
# and forget already-imported project modules so this session can't keep using stale code.
os.chdir("/content")
shutil.rmtree("/content/aps360_htr", ignore_errors=True)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
os.makedirs("/content/aps360_htr", exist_ok=True)
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall("/content/aps360_htr")
with open("/content/aps360_htr/CODE_VERSION", "w") as f:
    f.write("v16\n")
print("unpacked", len(zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).namelist()), "files | code v16")

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/__init__.py', 'src/data/__init__.py', 'src/utils/__init__.py', 'src/config.py', 'src/data/charset.py', 'src/data/transforms.py', 'src/data/iam.py', 'src/utils/metrics.py', 'src/data/recover.py', 'scripts/recover_labels.py', 'scripts/setup_iam.py', 'scripts/inspect_data.py']}):
    os.makedirs(f'/content/aps360_htr/{d}', exist_ok=True)
%cd /content/aps360_htr

### Package and settings

In [ ]:
%%writefile /content/aps360_htr/src/__init__.py
"""APS360 handwriting recognizer (CRNN + CTC)."""

In [ ]:
%%writefile /content/aps360_htr/src/data/__init__.py
"""Data loading: charset, IAM dataset, transforms."""

In [ ]:
%%writefile /content/aps360_htr/src/utils/__init__.py
"""Helpers such as CER/WER metrics."""

In [ ]:
%%writefile /content/aps360_htr/src/config.py
"""Central settings for the project.

Every script reads its defaults from here, so a change in one place applies
everywhere. Command-line flags in the scripts can override most of them.
"""
import os
from dataclasses import dataclass, field
from pathlib import Path

# Repo root = the folder that contains src/ (works on a laptop or in Colab).
ROOT = Path(__file__).resolve().parents[1]

# Where the IAM data lives. Set the HTR_DATA_ROOT environment variable (or pass
# --data-root to the scripts) to point somewhere else, e.g. in Colab:
#   os.environ["HTR_DATA_ROOT"] = "/content/iam"
#   or "/content/drive/MyDrive/APS360/iam" (works, but reading from Drive is slow)
DATA_ROOT = Path(os.environ.get("HTR_DATA_ROOT", ROOT / "data" / "iam"))


@dataclass
class DataConfig:
    # Where IAM lives. Expected layout (same for the fake stand-in data):
    #   <iam_root>/ascii/words.txt
    #   <iam_root>/ascii/forms.txt            (optional, gives writer IDs)
    #   <iam_root>/words/a01/a01-000u/a01-000u-00-00.png
    iam_root: Path = DATA_ROOT

    # Optional split lists (one ID per line: form, line or word ID).
    # Defaults to the Aachen splits (form IDs) that ship in data/splits/aachen.
    # Set split_dir=None to always use the automatic writer-disjoint split.
    split_dir: Path | None = ROOT / "data" / "splits" / "aachen"
    split_files: dict = field(default_factory=lambda: {
        "train": "train.uttlist",
        "val": "validation.uttlist",
        "test": "test.uttlist",
    })

    # Form -> writer table, used when the download has no ascii/forms.txt (e.g. Kaggle).
    # See data/iam_form_writers_SOURCE.md for where it comes from. None = don't use it.
    form_writers_csv: Path | None = ROOT / "data" / "iam_form_writers.csv"

    # Fallback split (used when split files are missing): fractions of items
    # per split, grouped so that no writer appears in two splits.
    split_fractions: tuple = (0.8, 0.1, 0.1)   # train, val, test
    split_seed: int = 42

    # Filtering
    drop_err: bool = True        # drop words whose segmentation status is "err"
    max_label_len: int = 32      # drop longer words (also keeps CTC happy)
    check_images: bool = True    # open every image once to drop broken/empty files

    # Image preprocessing
    img_height: int = 32         # brief: resize to 32 px high
    min_width: int = 16          # very short words (e.g. ".") get at least this
    max_width: int = 512         # very long words get squeezed to this

    # Loader
    batch_size: int = 32
    num_workers: int = 0         # 0 is safest on Windows/macOS; raise on Linux


@dataclass
class ModelConfig:
    """CRNN size. Saved inside every checkpoint so evaluate.py can rebuild it."""
    cnn_channels: tuple = (64, 128, 256, 256, 512)  # the 5 conv blocks
    lstm_hidden: int = 256       # per direction (BiLSTM output = 2x this)
    lstm_layers: int = 2
    dropout: float = 0.3


@dataclass
class TrainConfig:
    epochs: int = 40
    batch_size: int = 64
    lr: float = 1e-3             # peak learning rate (AdamW)
    weight_decay: float = 1e-4
    grad_clip: float = 5.0       # LSTMs + CTC can produce the odd huge gradient
    patience: int = 8            # stop if val CER hasn't improved for this many epochs
    lr_patience: int = 3         # halve the LR after this many epochs without improvement
    amp: bool = True             # mixed precision on GPU (faster on a T4)
    num_workers: int = 2         # Colab has 2 CPU cores
    preload: bool = True         # keep resized images in RAM (much faster epochs)
    train_frac: float = 1.0      # <1 trains on a random subset (quick sweeps)
    max_batches: int | None = None  # cap batches per epoch (smoke tests)
    seed: int = 42
    out_dir: Path = ROOT / "outputs" / "runs" / "crnn"

### Character set and CTC decoding
Blank = 0, so real characters start at 1. Greedy decoding takes the best class per time step, merges repeats, then drops blanks.

In [ ]:
%%writefile /content/aps360_htr/src/data/charset.py
"""Character set, label encoding, and greedy CTC decoding.

CTC needs one extra class, the "blank", which means "no character here".
We reserve index 0 for it, so real characters start at index 1.
"""
import string

# Letters, digits and the punctuation that appears in IAM word labels.
# (IAM words never contain a space, since each image is one word.)
DEFAULT_CHARS = (
    string.ascii_lowercase
    + string.ascii_uppercase
    + string.digits
    + "!\"#&'()*+,-./:;?"
)

BLANK = 0  # CTC blank index


class Charset:
    def __init__(self, chars: str = DEFAULT_CHARS):
        if len(set(chars)) != len(chars):
            raise ValueError("charset contains duplicate characters")
        self.chars = chars
        # char -> index, starting at 1 because 0 is the blank
        self.char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
        self.idx_to_char = {i + 1: c for i, c in enumerate(chars)}

    @property
    def num_classes(self) -> int:
        """Output size of the model's last layer (characters + blank)."""
        return len(self.chars) + 1

    def is_valid(self, text: str) -> bool:
        """True if every character of `text` is in the charset."""
        return all(c in self.char_to_idx for c in text)

    def encode(self, text: str) -> list[int]:
        """'cat' -> [3, 1, 20]. Raises KeyError on unknown characters."""
        return [self.char_to_idx[c] for c in text]

    def decode(self, indices) -> str:
        """Plain lookup, no CTC logic. Blanks (0) are skipped."""
        return "".join(self.idx_to_char[int(i)] for i in indices if int(i) != BLANK)

    def ctc_greedy_decode(self, indices) -> str:
        """Turn a per-frame best path into text.

        CTC rule: first merge repeated indices, then remove blanks.
        e.g. [0, 3, 3, 0, 1, 1, 20, 0] -> [3, 1, 20] -> 'cat'
        A blank between two equal letters keeps both: [12, 0, 12] -> 'll'.
        """
        out, prev = [], None
        for i in indices:
            i = int(i)
            if i != prev and i != BLANK:
                out.append(self.idx_to_char[i])
            prev = i
        return "".join(out)

    def decode_batch(self, log_probs) -> list[str]:
        """Greedy-decode model output of shape (T, B, C) into B strings.

        T = time steps (image columns after the CNN), B = batch, C = classes.
        This is the same layout nn.CTCLoss expects.
        """
        best = log_probs.argmax(dim=2)  # (T, B): most likely class per frame
        return [self.ctc_greedy_decode(best[:, b].tolist()) for b in range(best.shape[1])]

### Image preprocessing

In [ ]:
%%writefile /content/aps360_htr/src/data/transforms.py
"""Image preprocessing: resize to a fixed height and normalize.

Augmentation (webcam-style distortions, Model B) is in src/data/augment.py.
"""
import numpy as np
import torch
from PIL import Image

# After normalization, white paper = +1.0 and black ink = -1.0.
# The collate function pads with this value so padding looks like paper.
PAD_VALUE = 1.0


def resized_width(w: int, h: int, height: int, min_width: int, max_width: int) -> int:
    """Width after scaling an image of size (w, h) to `height` px, clamped."""
    return max(min_width, min(max_width, round(w * height / h)))


def resize_keep_aspect(img: Image.Image, height: int, min_width: int, max_width: int) -> Image.Image:
    """Scale a grayscale image to `height` px, keeping its width/height ratio.

    The width is clamped to [min_width, max_width]: tiny words are stretched a
    little and very long words are squeezed a little.
    """
    new_w = resized_width(*img.size, height, min_width, max_width)
    return img.resize((new_w, height), Image.Resampling.BILINEAR)


def to_tensor_normalized(img: Image.Image) -> torch.Tensor:
    """PIL grayscale image -> float tensor of shape (1, H, W) in [-1, 1]."""
    arr = np.asarray(img, dtype=np.float32) / 255.0   # [0, 1]
    arr = (arr - 0.5) / 0.5                            # [-1, 1]
    return torch.from_numpy(arr).unsqueeze(0)          # add channel dim


class Preprocess:
    """Resize + normalize. Optionally runs an augmentation first."""

    def __init__(self, height=32, min_width=16, max_width=512, augment=None):
        self.height, self.min_width, self.max_width = height, min_width, max_width
        self.augment = augment  # callable PIL -> PIL, or None

    def resize(self, img: Image.Image) -> Image.Image:
        return resize_keep_aspect(img, self.height, self.min_width, self.max_width)

    def resized_width(self, w: int, h: int) -> int:
        return resized_width(w, h, self.height, self.min_width, self.max_width)

    def __call__(self, img: Image.Image) -> torch.Tensor:
        if self.augment is not None:
            img = self.augment(img)
        return to_tensor_normalized(self.resize(img))


# Webcam-style augmentation (Model B, Step 6) lives in src/data/augment.py: it wraps the
# TRAINING dataset only, so validation/test (and Model A) keep using this clean Preprocess.

### IAM loading, filtering, splitting and batching

In [ ]:
%%writefile /content/aps360_htr/src/data/iam.py
"""IAM words: parse words.txt, filter, split by writer, and load as a Dataset.

words.txt line format (lines starting with '#' are comments):

    a01-000u-00-00 ok 154 408 768 27 51 AT A
    |              |  |   |   |   |  |  |  +-- transcription (the label)
    |              |  |   +---+---+--+  +----- grammatical tag
    |              |  |   bounding box x y w h
    |              |  +-- graylevel used to binarize
    |              +----- segmentation status: ok / err
    +-------------------- word ID = form-line-word

Image path for that ID:  words/a01/a01-000u/a01-000u-00-00.png
"""
import logging
import random
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

from src.config import DataConfig
from src.data.charset import Charset
from src.data.transforms import PAD_VALUE, Preprocess, to_tensor_normalized

log = logging.getLogger(__name__)

SPLITS = ("train", "val", "test")


@dataclass
class WordItem:
    word_id: str   # a01-000u-00-00
    form_id: str   # a01-000u
    writer: str | None   # writer ID (forms.txt or the bundled table), or None if unknown
    path: Path
    text: str


# ---------------------------------------------------------------- parsing

def word_id_parts(word_id: str):
    """'a01-000u-00-00' -> form 'a01-000u', line 'a01-000u-00'."""
    parts = word_id.split("-")
    return "-".join(parts[:2]), "-".join(parts[:3])


def image_path(root: Path, word_id: str) -> Path:
    form_id, _ = word_id_parts(word_id)
    return root / "words" / form_id.split("-")[0] / form_id / f"{word_id}.png"


def parse_words_txt(path: Path, malformed: list | None = None):
    """Yield (word_id, status, text) for every non-comment line.

    Lines with fewer than 9 fields can't be used (no label). They're skipped
    and appended to `malformed` so the caller can report them. In practice
    this happens when a words.txt file has been cut off part-way through.
    """
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip() or line.startswith("#"):
                continue
            fields = line.split(" ")
            if len(fields) < 9:
                log.warning("skipping malformed line: %r", line)
                if malformed is not None:
                    malformed.append(line)
                continue
            # The label is everything after the 8 fixed fields.
            yield fields[0], fields[1], " ".join(fields[8:])


def load_writers_csv(csv_path: Path) -> dict:
    """data/iam_form_writers.csv -> {form_id: writer_id}."""
    writers = {}
    with open(csv_path, encoding="utf-8") as f:
        next(f)  # header
        for line in f:
            if line.strip():
                form, writer = line.strip().split(",")
                writers[form] = writer
    return writers


def find_writers(cfg: DataConfig):
    """Return ({form_id: writer_id} or None, description of the source)."""
    forms_txt = Path(cfg.iam_root) / "ascii" / "forms.txt"
    if forms_txt.exists():
        return load_writers(forms_txt), "forms.txt"
    if cfg.form_writers_csv and Path(cfg.form_writers_csv).exists():
        return load_writers_csv(Path(cfg.form_writers_csv)), "bundled iam_form_writers.csv"
    return None, "unavailable"


def load_writers(forms_txt: Path) -> dict:
    """forms.txt -> {form_id: writer_id}. Line format: 'a01-000u 000 2 prt ...'."""
    writers = {}
    with open(forms_txt, encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.split()
            writers[fields[0]] = fields[1]
    return writers


# Word images that are known to be broken (empty/corrupt PNG) in the IAM release.
# They are always dropped and counted as "bad_or_missing_image".
KNOWN_BAD_WORD_IDS = frozenset({
    "a01-117-05-02",
    "r06-022-03-05",
})


def file_ok(path: Path) -> bool:
    """Cheap check (no decoding): the file exists and is not empty."""
    try:
        return path.stat().st_size > 0
    except OSError:
        return False


def image_ok(path: Path) -> bool:
    """False if the file is missing, empty, or not a readable image."""
    try:
        if path.stat().st_size == 0:
            return False
        with Image.open(path) as im:
            im.verify()  # cheap integrity check, doesn't decode all pixels
        with Image.open(path) as im:
            return im.size[0] > 0 and im.size[1] > 0
    except Exception:
        return False


def load_items(cfg: DataConfig, charset: Charset):
    """Read words.txt and apply all filters.

    Returns (items, drop_counts). drop_counts records how many words each
    filter removed, in the order the filters are applied; the report needs these.
    """
    root = Path(cfg.iam_root)
    words_txt = root / "ascii" / "words.txt"
    if not words_txt.exists():
        raise FileNotFoundError(f"{words_txt} not found. See README for the expected layout.")

    writers, writer_source = find_writers(cfg)
    if writers is None:
        log.warning("No writer IDs (no forms.txt or bundled table): writer checks are skipped")

    # Start every filter at 0 so the report table always lists all of them.
    drops = Counter({k: 0 for k in ("segmentation_err", "empty_label", "chars_outside_charset",
                                    "too_long", "bad_or_missing_image")})
    candidates, total = [], 0
    malformed = []
    for word_id, status, text in parse_words_txt(words_txt, malformed):
        total += 1
        if cfg.drop_err and status == "err":
            drops["segmentation_err"] += 1
            continue
        if not text:
            drops["empty_label"] += 1
            continue
        if not charset.is_valid(text):
            drops["chars_outside_charset"] += 1
            continue
        if len(text) > cfg.max_label_len:
            drops["too_long"] += 1
            continue
        form_id, _ = word_id_parts(word_id)
        writer = writers.get(form_id) if writers else None
        candidates.append(WordItem(word_id, form_id, writer, image_path(root, word_id), text))

    # Last filter: bad or missing images.
    #  - known broken files are always dropped;
    #  - check_images=True opens every image (thorough, ~1-2 min on all of IAM);
    #  - check_images=False only checks that each file exists and isn't empty
    #    (a few seconds). Anything still unreadable is dropped later, when the
    #    images are loaded (IAMWords.preload / drop_unreadable), never a crash.
    # Threads help because the time is spent waiting on the disk, not the CPU.
    known = [it for it in candidates if it.word_id in KNOWN_BAD_WORD_IDS]
    candidates = [it for it in candidates if it.word_id not in KNOWN_BAD_WORD_IDS]
    with ThreadPoolExecutor(max_workers=16) as pool:
        ok = list(pool.map(image_ok if cfg.check_images else file_ok, (it.path for it in candidates)))
    items = [it for it, good in zip(candidates, ok) if good]
    bad_ids = sorted([it.word_id for it in known] + [it.word_id for it, good in zip(candidates, ok) if not good])
    drops["bad_or_missing_image"] += len(bad_ids)
    if bad_ids:
        print(f"dropped {len(bad_ids)} bad or missing word image(s): {', '.join(bad_ids[:10])}"
              + (" ..." if len(bad_ids) > 10 else ""))

    drops["malformed_line"] = len(malformed)
    drops["total_lines"] = total
    drops["writer_source"] = writer_source
    drops["kept"] = len(items)
    drops["bad_image_ids"] = bad_ids
    return items, dict(drops)


# ---------------------------------------------------------------- splitting

def split_from_files(items, split_dir: Path, split_files: dict):
    """Assign items using ID lists (form, line or word IDs, one per line).

    Works with the Aachen splits (form IDs like 'a01-000u'). Items whose form
    isn't in any list are left out and counted as 'not_in_any_split'.
    """
    id_to_split = {}
    for split, fname in split_files.items():
        for line in open(Path(split_dir) / fname, encoding="utf-8"):
            if line.strip():
                id_to_split[line.strip()] = split

    out = {s: [] for s in SPLITS}
    missing = 0
    for it in items:
        form_id, line_id = word_id_parts(it.word_id)
        split = id_to_split.get(it.word_id) or id_to_split.get(line_id) or id_to_split.get(form_id)
        if split is None:
            missing += 1
        else:
            out[split].append(it)
    return out, missing


def writer_disjoint_split(items, fractions=(0.8, 0.1, 0.1), seed=42):
    """Deterministic split where each writer goes to exactly one split.

    If writer IDs are unknown, whole forms are grouped by form prefix (e.g. 'a01')
    instead. That keeps each form together but can't guarantee unseen writers.

    Writers are sorted, shuffled with a fixed seed, then handed to train until
    it holds ~80% of the words, then val, then test. Same data + same seed
    -> same split every time.
    """
    by_writer = defaultdict(list)
    for it in items:
        by_writer[it.writer or "prefix:" + it.form_id.split("-")[0]].append(it)
    writers = sorted(by_writer)
    random.Random(seed).shuffle(writers)

    n = len(items)
    cut_train = fractions[0] * n
    cut_val = (fractions[0] + fractions[1]) * n
    out = {s: [] for s in SPLITS}
    count = 0
    for w in writers:
        split = "train" if count < cut_train else "val" if count < cut_val else "test"
        out[split].extend(by_writer[w])
        count += len(by_writer[w])
    return out


def build_splits(cfg: DataConfig, charset: Charset):
    """Load, filter and split IAM. Returns (splits_dict, stats_dict).

    Uses the split files if they all exist and match at least one word;
    otherwise falls back to the automatic writer-disjoint split.
    """
    items, stats = load_items(cfg, charset)
    splits = None

    sd = Path(cfg.split_dir) if cfg.split_dir else None
    if sd and all((sd / f).exists() for f in cfg.split_files.values()):
        splits, missing = split_from_files(items, sd, cfg.split_files)
        if all(splits[s] for s in SPLITS):
            stats["split_method"] = f"split files in {sd}"
            stats["not_in_any_split"] = missing
        else:
            log.warning("Split files left a split empty (IDs don't match this data); "
                        "using writer-disjoint split instead")
            splits = None

    if splits is None:
        splits = writer_disjoint_split(items, cfg.split_fractions, cfg.split_seed)
        stats["split_method"] = f"writer-disjoint (seed={cfg.split_seed})"

    log.info("IAM filtering: %s", stats)
    for s in SPLITS:
        log.info("%s: %d words", s, len(splits[s]))
    return splits, stats


# ---------------------------------------------------------------- Dataset

class IAMWords(Dataset):
    """Returns (image (1,32,W), encoded target (L,), target length, text)."""

    def __init__(self, items, charset: Charset, transform: Preprocess):
        self.items, self.charset, self.transform = items, charset, transform
        self._cache = None   # filled by preload(): resized uint8 images
        self.dropped = []    # word IDs removed because their image couldn't be read
        self.skipped = set() # word IDs skipped on the fly in __getitem__ (not preloaded)

    def __len__(self):
        return len(self.items)

    def _drop(self, bad_idx, reason):
        """Remove items whose image could not be read (and remember them)."""
        if not bad_idx:
            return
        bad = set(bad_idx)
        ids = [self.items[i].word_id for i in sorted(bad)]
        self.dropped += ids
        self.items = [it for i, it in enumerate(self.items) if i not in bad]
        if self._cache is not None:
            self._cache = [a for i, a in enumerate(self._cache) if i not in bad]
        msg = (f"dropped {len(ids)} unreadable image(s) while {reason}: {', '.join(ids[:10])}"
               + (" ..." if len(ids) > 10 else ""))
        print(msg)

    def _load_resized(self, it):
        """Resized uint8 image, or None if the file can't be decoded."""
        try:
            with Image.open(it.path) as im:
                img = im.convert("L")
            if img.size[0] == 0 or img.size[1] == 0:
                return None
            return np.asarray(self.transform.resize(img), dtype=np.uint8)
        except Exception:
            return None

    def preload(self, num_threads: int = 8):
        """Load and resize every image once and keep it in RAM (uint8).

        Decoding PNGs is the slowest part of training on Colab (only 2 CPU
        cores), so this makes every epoch after the first much faster.
        ~100k IAM words at 32 px high take roughly 300 MB.
        Images that can't be decoded are dropped (listed in self.dropped).
        Not used when an augmentation is set, since that must run on the
        original image each time (then use drop_unreadable()).
        """
        if self.transform.augment is not None:
            log.warning("preload skipped: augmentation is on")
            return self.drop_unreadable(num_threads)
        with ThreadPoolExecutor(max_workers=num_threads) as pool:
            cache = list(pool.map(self._load_resized, self.items))
        self._cache = cache
        self._drop([i for i, a in enumerate(cache) if a is None], "preloading")
        return self

    def drop_unreadable(self, num_threads: int = 8):
        """Decode every image once (without keeping it) and drop the unreadable ones."""
        with ThreadPoolExecutor(max_workers=num_threads) as pool:
            ok = list(pool.map(lambda it: image_ok(it.path), self.items))
        self._drop([i for i, good in enumerate(ok) if not good], "checking images")
        return self

    def widths(self, num_threads: int = 8) -> list[int]:
        """Width of each image after resizing (used to batch similar widths).
        An unreadable file gets the minimum width here; __getitem__ skips it."""
        if self._cache is not None:
            return [a.shape[1] for a in self._cache]
        def width(it):
            try:
                with Image.open(it.path) as im:   # only reads the header
                    w, h = im.size
                return self.transform.resized_width(w, h)
            except Exception:
                return self.transform.min_width
        with ThreadPoolExecutor(max_workers=num_threads) as pool:
            return list(pool.map(width, self.items))

    def __getitem__(self, idx):
        """Returns None for an unreadable image (collate_fn leaves it out of the batch)."""
        it = self.items[idx]
        if self._cache is not None:
            x = to_tensor_normalized(Image.fromarray(self._cache[idx]))
        else:
            try:
                with Image.open(it.path) as im:
                    img = im.convert("L")  # grayscale
                x = self.transform(img)
            except Exception as e:
                if it.word_id not in self.skipped:
                    self.skipped.add(it.word_id)
                    msg = f"skipping unreadable image {it.word_id} ({type(e).__name__})"
                    print(msg)
                return None
        target = torch.tensor(self.charset.encode(it.text), dtype=torch.long)
        return x, target, len(target), it.text


def collate_fn(batch):
    """Pad images on the right to the widest one; pack targets for CTC.

    Returns a dict:
      images         (B, 1, H, W_max)  padded with paper-white
      widths         (B,)  original widths, to compute CTC input lengths later
      targets        (sum of lengths,) all labels concatenated (nn.CTCLoss format)
      target_lengths (B,)
      texts          list of B strings
    """
    batch = [b for b in batch if b is not None]     # unreadable images were returned as None
    if not batch:
        return None
    images, targets, lengths, texts = zip(*batch)
    h = images[0].shape[1]
    w_max = max(im.shape[2] for im in images)
    padded = torch.full((len(images), 1, h, w_max), PAD_VALUE)
    for i, im in enumerate(images):
        padded[i, :, :, : im.shape[2]] = im
    return {
        "images": padded,
        "widths": torch.tensor([im.shape[2] for im in images], dtype=torch.long),
        "targets": torch.cat(targets),
        "target_lengths": torch.tensor(lengths, dtype=torch.long),
        "texts": list(texts),
    }


def make_datasets(cfg: DataConfig | None = None, charset: Charset | None = None, train_augment=None):
    """One-call helper: returns ({'train': ds, 'val': ds, 'test': ds}, stats)."""
    cfg = cfg or DataConfig()
    charset = charset or Charset()
    splits, stats = build_splits(cfg, charset)
    base = dict(height=cfg.img_height, min_width=cfg.min_width, max_width=cfg.max_width)
    ds = {
        s: IAMWords(splits[s], charset,
                    Preprocess(**base, augment=train_augment if s == "train" else None))
        for s in SPLITS
    }
    return ds, stats

### Metrics (CER / WER)

In [ ]:
%%writefile /content/aps360_htr/src/utils/metrics.py
"""Character and word error rate (CER / WER).

Both are corpus-level: add up the edits over all samples, then divide by the
total length of the references. This weights long words more than short ones,
which is the standard way to report CER/WER (averaging per-word rates is not).
"""
try:
    import editdistance as _ed  # fast C implementation, if installed
except ImportError:  # pragma: no cover
    _ed = None


def edit_distance(a, b) -> int:
    """Levenshtein distance: fewest insertions, deletions and substitutions
    turning sequence `a` into `b`. Works on strings or lists of words."""
    if _ed is not None:
        return _ed.eval(a, b)
    return _edit_distance_py(a, b)


def _edit_distance_py(a, b) -> int:
    """Pure-Python dynamic programming version (one row at a time)."""
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        cur = [i]
        for j, y in enumerate(b, start=1):
            cur.append(min(
                prev[j] + 1,             # delete x
                cur[j - 1] + 1,          # insert y
                prev[j - 1] + (x != y),  # substitute (free if equal)
            ))
        prev = cur
    return prev[-1]


def cer(refs: list[str], hyps: list[str]) -> float:
    """Total character edits / total reference characters."""
    assert len(refs) == len(hyps), "need one prediction per reference"
    edits = sum(edit_distance(r, h) for r, h in zip(refs, hyps))
    total = sum(len(r) for r in refs)
    return edits / max(total, 1)


def wer(refs: list[str], hyps: list[str]) -> float:
    """Total word edits / total reference words (words split on whitespace).

    For our single-word images this is just the fraction of words that are
    not exactly right.
    """
    assert len(refs) == len(hyps), "need one prediction per reference"
    edits = sum(edit_distance(r.split(), h.split()) for r, h in zip(refs, hyps))
    total = sum(len(r.split()) for r in refs)
    return edits / max(total, 1)

### Label recovery from IAM's embedded XML

In [ ]:
%%writefile /content/aps360_htr/src/data/recover.py
"""Recover IAM labels from the metadata embedded in the original IAM PNG files.

Every original IAM word PNG carries its form's XML in a PNG text chunk (key 'xml'
or 'Description'). The XML holds the word's ID (in select="...@id='a01-000u-00-00'"),
its text, grammatical tag and bounding box, plus the line's segmentation status
(ok/err) and binarisation threshold. Those are exactly the fields of a words.txt
line, so a missing words.txt line can be rebuilt from the image alone:

    a01-000u-00-00 ok 154 408 768 27 51 AT A
    id             |  |   x   y   w  h  tag text
                   |  threshold
                   line segmentation status

Used by scripts/recover_labels.py. Sources of PNGs: the user's own images (if they
still carry the XML), or the Hugging Face copy of IAM (priyank-m/IAM_words_text_recognition),
whose PNGs are the originals with the XML intact.
"""
import hashlib
import io
import re
import xml.etree.ElementTree as ET

from PIL import Image

_SELECT_RE = re.compile(r"@id=(?:'|&apos;)([a-z]\d\d-\d{3}[a-z]?-\d\d-\d\d)(?:'|&apos;)")
_DOCTYPE_RE = re.compile(r"<!DOCTYPE[^>]*>")


def metadata_xml(img: Image.Image) -> str | None:
    """The embedded IAM XML of an opened PNG, or None."""
    for key in ("xml", "Description", "XML:com.adobe.xmp", "Comment"):
        v = img.info.get(key)
        if isinstance(v, bytes):
            v = v.decode("utf-8", "ignore")
        if v and "<form" in v:
            return v
    return None


def entry_from_xml(xml: str) -> dict | None:
    """Parse one image's XML into the fields of its words.txt line."""
    m = _SELECT_RE.search(xml)
    if not m:
        return None
    wid = m.group(1)
    try:
        root = ET.fromstring(_DOCTYPE_RE.sub("", xml))
    except ET.ParseError:
        return None
    for line in root.iter("line"):
        for w in line.iter("word"):
            if w.get("id") != wid:
                continue
            cmps = w.findall("cmp")
            if cmps:
                xs = [int(c.get("x")) for c in cmps]; ys = [int(c.get("y")) for c in cmps]
                x2 = [int(c.get("x")) + int(c.get("width")) for c in cmps]
                y2 = [int(c.get("y")) + int(c.get("height")) for c in cmps]
                box = (min(xs), min(ys), max(x2) - min(xs), max(y2) - min(ys))
            else:
                box = (-1, -1, -1, -1)
            return {"id": wid, "status": line.get("segmentation", "ok"),
                    "threshold": line.get("threshold", "0"), "box": box,
                    "tag": w.get("tag", "UNK"), "text": w.get("text", ""),
                    "writer": root.get("writer-id")}
    return None


def entry_from_png_bytes(data: bytes) -> dict | None:
    with Image.open(io.BytesIO(data)) as im:   # text chunks are read without decoding pixels
        xml = metadata_xml(im)
    return entry_from_xml(xml) if xml else None


def entry_from_png_path(path) -> dict | None:
    try:
        with Image.open(path) as im:
            xml = metadata_xml(im)
        return entry_from_xml(xml) if xml else None
    except Exception:
        return None


def words_txt_line(e: dict) -> str:
    x, y, w, h = e["box"]
    return f"{e['id']} {e['status']} {e['threshold']} {x} {y} {w} {h} {e['tag']} {e['text']}"


def pixel_hash(img: Image.Image) -> str:
    """Hash of the decoded grayscale pixels + size: equal for identical images even if
    the PNG was re-compressed or its metadata stripped."""
    g = img.convert("L")
    return hashlib.sha1(f"{g.size}".encode() + g.tobytes()).hexdigest()

In [ ]:
%%writefile /content/aps360_htr/scripts/recover_labels.py
"""Rebuild the missing lines of a cut-off words.txt.

Some IAM copies (e.g. the Kaggle one) have every word image but a words.txt that
stops part-way (at e07-072). This script writes words_recovered.txt: all the
original lines, plus a rebuilt line for every image that had no label.

Where the labels come from (see src/data/recover.py):
  1. the IAM XML embedded in your own PNGs, if they still have it (no download), else
  2. the Hugging Face copy of IAM (priyank-m/IAM_words_text_recognition, ~1.2 GB
     parquet), whose original PNGs carry the same XML. Entries are matched by
     word ID. Any image whose ID isn't found is matched by a hash of its pixels.

Checks printed for the report:
  - rebuilt vs original lines for words that have both (should be ~100% identical)
  - pixel check: HF image == your image for a sample of recovered words
  - matched / unmatched / ambiguous counts. Below --min-rate (90%) nothing is
    written, and you keep training on the labelled subset.

  python scripts/recover_labels.py --data-root /content/iam --out words_recovered.txt
"""
import argparse
import json
import random
import sys
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from PIL import Image

from src.data.iam import image_path, parse_words_txt
from src.data.recover import entry_from_png_bytes, entry_from_png_path, pixel_hash, words_txt_line

HF_REPO = "priyank-m/IAM_words_text_recognition"
HF_FILES = ["data/train-00000-of-00001-bfc7b63751c36ab0.parquet",
            "data/val-00000-of-00001-472467affea948eb.parquet",
            "data/test-00000-of-00001-4cae70e6a03872e2.parquet"]


def list_image_ids(words_dir: Path) -> dict:
    """{word_id: path} for every PNG under words/ (follows the setup symlinks)."""
    import os
    ids = {}
    for d, _, files in os.walk(words_dir, followlinks=True):
        for f in files:
            if f.endswith(".png"):
                ids[f[:-4]] = Path(d) / f
    return ids


def download_hf(cache_dir: Path) -> list[Path]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    try:
        from huggingface_hub import hf_hub_download
        for f in HF_FILES:
            paths.append(Path(hf_hub_download(HF_REPO, f, repo_type="dataset", local_dir=str(cache_dir))))
        return paths
    except ImportError:
        pass
    for f in HF_FILES:
        p = cache_dir / Path(f).name
        if not p.exists():
            print(f"  downloading {f} ...")
            urllib.request.urlretrieve(f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{f}", p)
        paths.append(p)
    return paths


def scan_hf(parquets, wanted: set, sample_ids: set, threads=8):
    """One pass over the HF parquet files.
    Returns ({id: entry} for wanted ids, {id: png bytes} for sample ids, rows without an id)."""
    import pyarrow.parquet as pq
    found, sample_bytes, no_id = {}, {}, []
    for p in parquets:
        pf = pq.ParquetFile(p)
        for batch in pf.iter_batches(batch_size=2000, columns=["image", "text"]):
            imgs = batch.column("image").to_pylist()
            texts = batch.column("text").to_pylist()
            with ThreadPoolExecutor(threads) as pool:
                entries = list(pool.map(lambda d: entry_from_png_bytes(d["bytes"]), imgs))
            for e, d, t in zip(entries, imgs, texts):
                if e is None:
                    no_id.append((d["bytes"], t))
                    continue
                if e["id"] in wanted:
                    found[e["id"]] = e
                if e["id"] in sample_ids:
                    sample_bytes[e["id"]] = d["bytes"]
        print(f"  scanned {Path(p).name}: {len(found)} wanted ids found so far")
    return found, sample_bytes, no_id


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", required=True, help="folder with ascii/words.txt and words/")
    ap.add_argument("--words-txt", default=None, help="default: <data-root>/ascii/words.txt")
    ap.add_argument("--out", required=True, help="where to write words_recovered.txt")
    ap.add_argument("--hf-cache", default="/content/hf_iam", help="where to put the HF parquet files")
    ap.add_argument("--hf-parquet", nargs="*", default=None, help="use these local parquet files")
    ap.add_argument("--no-local-xml", action="store_true", help="skip reading XML from your own PNGs")
    ap.add_argument("--min-rate", type=float, default=0.90)
    ap.add_argument("--verify-sample", type=int, default=3000)
    ap.add_argument("--report", default=None, help="also save the numbers as JSON here")
    ap.add_argument("--preview", default=None, help="save a picture of recovered words here")
    ap.add_argument("--threads", type=int, default=16)
    a = ap.parse_args()
    t0 = time.time()
    root = Path(a.data_root)
    words_txt = Path(a.words_txt or root / "ascii" / "words.txt")

    malformed = []
    original = {wid: (st, txt) for wid, st, txt in parse_words_txt(words_txt, malformed)}
    original_lines = [l.rstrip("\n") for l in open(words_txt, encoding="utf-8")
                      if l.strip() and not l.startswith("#") and len(l.split(" ")) >= 9]
    images = list_image_ids(root / "words")
    unlabeled = sorted(set(images) - set(original))
    print(f"words.txt: {len(original)} labelled words ({len(malformed)} cut-off line(s)); "
          f"images: {len(images)}; images without a label: {len(unlabeled)}")
    report = {"labelled_original": len(original), "images": len(images), "unlabeled": len(unlabeled)}
    if not unlabeled:
        print("Nothing to recover: every image already has a label.")
        return

    recovered, source = {}, None
    # ---- 1. XML inside the user's own PNGs (free, no download)
    if not a.no_local_xml:
        probe = unlabeled[:200]
        with ThreadPoolExecutor(a.threads) as pool:
            hits = [e for e in pool.map(lambda i: entry_from_png_path(images[i]), probe) if e]
        print(f"embedded XML in your own images: {len(hits)}/{len(probe)} probed")
        if len(hits) >= 0.9 * len(probe):
            with ThreadPoolExecutor(a.threads) as pool:
                for wid, e in zip(unlabeled, pool.map(lambda i: entry_from_png_path(images[i]), unlabeled)):
                    if e and e["id"] == wid:
                        recovered[wid] = e
            source = "embedded XML in the user's PNGs"

    # ---- 2. Hugging Face copy of IAM, matched by word ID (pixel hash as backup)
    sample_ids = set(random.Random(0).sample(unlabeled, min(a.verify_sample, len(unlabeled))))
    hf_sample = {}
    if len(recovered) < a.min_rate * len(unlabeled):
        recovered.clear()
        parquets = [Path(p) for p in a.hf_parquet] if a.hf_parquet else download_hf(Path(a.hf_cache))
        # also fetch the originally-labelled ids, to check rebuilt lines against real ones
        check_ids = set(random.Random(1).sample(sorted(original), min(5000, len(original))))
        found, hf_sample, no_id = scan_hf(parquets, set(unlabeled) | check_ids, sample_ids, a.threads)
        recovered = {i: found[i] for i in unlabeled if i in found}
        source = f"Hugging Face {HF_REPO} (matched by word ID)"

        # check rebuilt lines against the original words.txt lines
        orig_line = {l.split(" ")[0]: l for l in original_lines}
        both = [i for i in check_ids if i in found and i in orig_line]
        same = sum(words_txt_line(found[i]) == orig_line[i] for i in both)
        same_label = sum(found[i]["text"] == original[i][1] and found[i]["status"] == original[i][0] for i in both)
        print(f"rebuilt vs original words.txt ({len(both)} words): identical lines {same/len(both):.2%}, "
              f"same text+status {same_label/len(both):.2%}" if both else "no overlap to check")
        report.update(check_words=len(both), identical_lines=same, same_text_status=same_label)

        # backup: pixel-hash match for images whose id wasn't found (expected: none)
        missing = [i for i in unlabeled if i not in recovered]
        if missing and no_id:
            print(f"{len(missing)} ids not found; trying pixel hashes against {len(no_id)} HF rows without ids")
            by_hash = {}
            for data, text in no_id:
                h = pixel_hash(Image.open(__import__("io").BytesIO(data)))
                by_hash.setdefault(h, []).append(text)
            amb = 0
            for i in missing:
                texts = by_hash.get(pixel_hash(Image.open(images[i])), [])
                if len(set(texts)) == 1:
                    recovered[i] = {"id": i, "status": "ok", "threshold": "0", "box": (-1, -1, -1, -1),
                                    "tag": "RECOVERED", "text": texts[0]}
                elif texts:
                    amb += 1
            report["ambiguous"] = amb

    # ---- pixel check: is the HF image really the same picture as ours?
    if hf_sample:
        import io
        def same(i):
            try:
                return pixel_hash(Image.open(io.BytesIO(hf_sample[i]))) == pixel_hash(Image.open(images[i]))
            except Exception:
                return False
        with ThreadPoolExecutor(a.threads) as pool:
            res = list(pool.map(same, sorted(hf_sample)))
        rate = sum(res) / len(res)
        print(f"pixel check on {len(res)} recovered words: {rate:.2%} identical images")
        report.update(pixel_checked=len(res), pixel_identical=sum(res))

    n = len(recovered)
    rate = n / len(unlabeled)
    report.update(source=source, matched=n, unmatched=len(unlabeled) - n, match_rate=round(rate, 4),
                  ambiguous=report.get("ambiguous", 0))
    print(f"\nmatched {n} / {len(unlabeled)} unlabeled images ({rate:.2%}); "
          f"unmatched {len(unlabeled) - n}; ambiguous {report['ambiguous']}; source: {source}")
    if a.report:
        Path(a.report).write_text(json.dumps(report, indent=2))

    if rate < a.min_rate:
        print(f"\nMatch rate is below {a.min_rate:.0%}, so NO words_recovered.txt was written. "
              "Training will use the labelled subset only.")
        sys.exit(2)

    out = Path(a.out)
    with open(out, "w", encoding="utf-8") as f:
        f.write("#--- words_recovered.txt: IAM words.txt format ---\n"
                f"# original lines from {words_txt.name}, then {n} lines rebuilt from IAM's embedded XML\n"
                f"# source of rebuilt lines: {source}\n"
                "# format: a01-000u-00-00 ok 154 408 768 27 51 AT A\n")
        f.write("\n".join(original_lines) + "\n")
        f.write("#--- recovered entries below (one per image that had no label) ---\n")
        f.write("\n".join(words_txt_line(recovered[i]) for i in sorted(recovered)) + "\n")
    out.with_name(out.stem + "_ids.txt").write_text("\n".join(sorted(recovered)) + "\n")
    print(f"wrote {out} ({len(original_lines) + n} words) in {time.time() - t0:.0f}s")

    if a.preview:
        import matplotlib; matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        pick = random.Random(2).sample(sorted(recovered), min(12, n))
        fig, axes = plt.subplots(len(pick), 1, figsize=(5, 0.9 * len(pick) + 0.4))
        for ax, i in zip(axes if len(pick) > 1 else [axes], pick):
            ax.imshow(Image.open(images[i]).convert("L"), cmap="gray")
            ax.set_title(f"{i}: {recovered[i]['text']!r} ({recovered[i]['status']})", fontsize=8, loc="left")
            ax.set_xticks([]); ax.set_yticks([])
        fig.tight_layout(); fig.savefig(a.preview, dpi=110); plt.close(fig)
        print(f"saved preview of recovered labels to {a.preview}")


if __name__ == "__main__":
    main()

### Scripts

In [ ]:
%%writefile /content/aps360_htr/scripts/setup_iam.py
"""Turn whatever IAM download you have into the layout the code expects.

Target layout (DATA_ROOT):
    <out>/ascii/words.txt          (+ forms.txt if your download has it)
    <out>/words/a01/a01-000u/a01-000u-00-00.png

Two ways to use it:

1) Auto-detect inside an unpacked folder (official or Kaggle download):
       python scripts/setup_iam.py --raw /content/iam_raw --out /content/iam
   It unpacks any .tgz / .tar.gz it finds, locates words.txt, and finds the
   folder of word images. It links them into <out> with symlinks, so nothing
   is copied. Images stored in one flat folder also work: each gets its own link.

2) The two official archives:
       python scripts/setup_iam.py --words-tgz words.tgz --ascii-tgz ascii.tgz --out data/iam
"""
import argparse
import os
import re
import shutil
import tarfile
from pathlib import Path

# IAM word IDs look like a01-000u-00-00 (form prefix, form, line, word)
WORD_ID_RE = re.compile(r"^[a-z]\d\d-\d{3}[a-z]?-\d\d-\d\d$")


# ------------------------------------------------------------ helpers

def safe_extract(tgz: Path, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tgz) as tf:
        try:
            tf.extractall(dest, filter="data")   # safe mode (Python 3.12+)
        except TypeError:
            tf.extractall(dest)                  # older Python


def extract_nested_archives(raw: Path):
    """Unpack every .tgz/.tar.gz under `raw` once (a marker file remembers)."""
    archives = [p for p in raw.rglob("*") if p.is_file() and p.name.endswith((".tgz", ".tar.gz"))]
    for a in archives:
        dest = a.parent / (a.name.split(".")[0] + "_extracted")
        if (dest / ".done").exists():
            continue
        print(f"  extracting {a} -> {dest}")
        safe_extract(a, dest)
        (dest / ".done").touch()


def count_word_lines(path: Path, max_lines=2000) -> int:
    """How many of the first lines look like IAM words.txt entries."""
    n = 0
    with open(path, encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            parts = line.split(" ")
            if len(parts) >= 9 and WORD_ID_RE.match(parts[0]) and parts[1] in ("ok", "err"):
                n += 1
    return n


def find_words_txt(raw: Path) -> Path:
    candidates = [p for p in raw.rglob("*.txt") if p.is_file()]
    scored = sorted(((count_word_lines(p), p) for p in candidates), reverse=True)
    if not scored or scored[0][0] == 0:
        raise FileNotFoundError(f"No IAM-style words.txt found under {raw}")
    # prefer a file literally called words.txt among equally good ones
    best_score = scored[0][0]
    best = [p for s, p in scored if s == best_score]
    return next((p for p in best if p.name == "words.txt"), best[0])


def find_image_root(raw: Path):
    """Return ('nested', root) if images sit at root/a01/a01-000u/<id>.png,
    else ('flat', None) when they don't follow that folder structure."""
    for png in raw.rglob("*.png"):
        wid = png.stem
        if not WORD_ID_RE.match(wid):
            continue
        form = "-".join(wid.split("-")[:2])
        if png.parent.name == form and png.parent.parent.name == form.split("-")[0]:
            return "nested", png.parent.parent.parent
        return "flat", None
    raise FileNotFoundError(f"No IAM word images (e.g. a01-000u-00-00.png) found under {raw}")


def link(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    os.symlink(src.resolve(), dst)


# ------------------------------------------------------------ main modes

def setup_from_raw(raw: Path, out: Path, words_txt: Path | None = None):
    raw = Path(raw)
    extract_nested_archives(raw)

    # A words.txt you pass in (e.g. the full official one) wins over the one in the download.
    words_txt = Path(words_txt) if words_txt else find_words_txt(raw)
    print(f"  words.txt   : {words_txt}")
    link(words_txt, out / "ascii" / "words.txt")
    forms = [p for p in raw.rglob("forms.txt") if p.is_file()]
    if forms:
        print(f"  forms.txt   : {forms[0]}")
        link(forms[0], out / "ascii" / "forms.txt")
    else:
        print("  forms.txt   : not found (fine: the Aachen splits don't need it)")

    kind, root = find_image_root(raw)
    if kind == "nested":
        print(f"  image root  : {root} (nested a01/a01-000u/... layout)")
        link(root, out / "words")
    else:
        print("  images are not in a01/a01-000u/ folders: linking each one into place")
        n = 0
        for png in raw.rglob("*.png"):
            wid = png.stem
            if WORD_ID_RE.match(wid):
                form = "-".join(wid.split("-")[:2])
                link(png, out / "words" / form.split("-")[0] / form / png.name)
                n += 1
        print(f"  linked {n} images")


def setup_from_tgz(words_tgz: Path, ascii_tgz: Path, out: Path):
    raw = out.parent / (out.name + "_raw")
    safe_extract(Path(ascii_tgz), raw / "ascii")
    safe_extract(Path(words_tgz), raw / "words")
    setup_from_raw(raw, out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", help="folder with an unpacked IAM download (any layout)")
    ap.add_argument("--words-tgz")
    ap.add_argument("--ascii-tgz")
    ap.add_argument("--words-txt", help="use this words.txt instead of the one in the download")
    ap.add_argument("--out", default="data/iam")
    a = ap.parse_args()
    out = Path(a.out)
    if a.raw:
        setup_from_raw(Path(a.raw), out, a.words_txt)
    elif a.words_tgz and a.ascii_tgz:
        setup_from_tgz(Path(a.words_tgz), Path(a.ascii_tgz), out)
    else:
        ap.error("give --raw, or both --words-tgz and --ascii-tgz")
    # os.walk with followlinks=True also counts images behind the symlinks
    n = sum(len([f for f in fs if f.endswith(".png")]) for _, _, fs in os.walk(out / "words", followlinks=True))
    print(f"IAM ready in {out.resolve()}: {n} word images")
    check_words_txt(out / "ascii" / "words.txt", n)


def check_words_txt(words_txt: Path, n_images: int):
    """Warn if words.txt labels far fewer words than there are images
    (e.g. a words.txt that was cut off part-way through)."""
    n_labels, last = 0, ""
    with open(words_txt, encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                n_labels += 1
                last = line.rstrip("\n")
    print(f"words.txt lists {n_labels} words; last line: {last!r}")
    if len(last.split(" ")) < 9:
        print("WARNING: the last line of words.txt is incomplete, so the file was cut off. "
              "Words after it have no labels and will be skipped.")
    if n_images > 1.1 * n_labels:
        print(f"WARNING: {n_images - n_labels} images have no line in words.txt. "
              "Get the full words.txt (inside IAM's official ascii.tgz) to use all of them.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/aps360_htr/scripts/inspect_data.py
"""Print dataset statistics and save a picture of one padded batch.

Usage:
  python scripts/inspect_data.py --data-root data/fake_iam
  python scripts/inspect_data.py --data-root /content/iam --out outputs/iam_batch.png
"""
import argparse
import json
import logging
import statistics
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import matplotlib
matplotlib.use("Agg")  # save to file, no window needed
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.config import DataConfig
from src.data.charset import Charset
from src.data.iam import SPLITS, collate_fn, make_datasets


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default=None, help="IAM folder (default: config.DATA_ROOT)")
    ap.add_argument("--split-dir", default=None, help="folder with split lists; 'none' = auto split")
    ap.add_argument("--split", default="train", choices=SPLITS)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--out", default="outputs/sample_batch.png")
    ap.add_argument("--stats-out", default=None, help="also save the stats as JSON here")
    ap.add_argument("--no-check-images", action="store_true", help="skip opening every image (faster)")
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

    cfg = DataConfig()
    if args.data_root:
        cfg.iam_root = Path(args.data_root)
    if args.split_dir:
        cfg.split_dir = None if args.split_dir.lower() == "none" else Path(args.split_dir)
    cfg.check_images = not args.no_check_images
    charset = Charset()

    datasets, stats = make_datasets(cfg, charset)

    print("\n=== Filtering (numbers for the report) ===")
    for k, v in stats.items():
        print(f"  {k:24s} {v}")

    print("\n=== Splits ===")
    writer_sets, unknown = {}, 0
    has_writers = any(i.writer is not None for s in SPLITS for i in datasets[s].items)
    for s in SPLITS:
        items = datasets[s].items
        writer_sets[s] = {i.writer for i in items if i.writer is not None}
        unknown += sum(i.writer is None for i in items)
        lens = [len(i.text) for i in items] or [0]
        n_w = f"{len(writer_sets[s]):4d}" if has_writers else "   ?"
        print(f"  {s:5s}: {len(items):6d} words | {n_w} writers | "
              f"label len mean {statistics.mean(lens):.1f}, max {max(lens)}")

    # Writer-disjoint check: no writer may appear in two splits.
    overlap = None
    if not has_writers:
        print("  writer IDs unavailable; Aachen splits are writer-independent by design"
              if "split files" in str(stats.get("split_method")) else
              "  writer IDs unavailable; the split groups whole forms by prefix instead")
    else:
        overlap = (writer_sets["train"] & writer_sets["val"]) | (writer_sets["train"] & writer_sets["test"]) \
            | (writer_sets["val"] & writer_sets["test"])
        print(f"  writer IDs from: {stats.get('writer_source')}")
        print(f"  writers shared between splits: {len(overlap)} (should be 0)")
        if unknown:
            print(f"  ({unknown} words have no writer ID and are left out of this check)")

    if stats.get("malformed_line"):
        print(f"\n  NOTE: {stats['malformed_line']} line(s) of words.txt had no label and were skipped.")
        print("  If it's the last line, your words.txt was cut off: every word after it is missing.")

    if args.stats_out:
        summary = dict(stats, split_sizes={s: len(datasets[s]) for s in SPLITS},
                       split_writers={s: len(writer_sets[s]) for s in SPLITS},
                       writers_shared_between_splits=None if overlap is None else len(overlap))
        Path(args.stats_out).parent.mkdir(parents=True, exist_ok=True)
        Path(args.stats_out).write_text(json.dumps(summary, indent=2))
        print(f"  stats saved to {Path(args.stats_out).resolve()}")

    train_chars = Counter("".join(i.text for i in datasets["train"].items))
    print(f"\n  charset size: {len(charset.chars)} chars + blank = {charset.num_classes} classes")
    print(f"  most common train chars: {train_chars.most_common(10)}")
    unused = [c for c in charset.chars if c not in train_chars]
    print(f"  charset chars never seen in train: {''.join(unused) or '(none)'}")

    ds = datasets[args.split]
    if len(ds) == 0:
        print(f"\n{args.split} split is empty; nothing to plot.")
        return
    widths = [ds[i][0].shape[2] for i in range(min(len(ds), 500))]
    print(f"\n  {args.split} image widths after resize (first {len(widths)}): "
          f"min {min(widths)}, median {int(statistics.median(widths))}, max {max(widths)}")

    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn,
                        generator=torch.Generator().manual_seed(0))
    batch = next(iter(loader))
    print("\n=== One batch ===")
    print(f"  images         {tuple(batch['images'].shape)}  (B, C, H, W_max)")
    print(f"  widths         {batch['widths'].tolist()}")
    print(f"  targets        {tuple(batch['targets'].shape)}  (all labels concatenated)")
    print(f"  target_lengths {batch['target_lengths'].tolist()}")
    print(f"  texts          {batch['texts']}")
    print(f"  decode check   {charset.decode(batch['targets'][:batch['target_lengths'][0]].tolist())!r}"
          f" == {batch['texts'][0]!r}")

    # Grid: each row is one padded image with its label as the title.
    n = len(batch["texts"])
    fig, axes = plt.subplots(n, 1, figsize=(6, 0.8 * n + 0.5))
    axes = [axes] if n == 1 else axes
    for ax, img, text, w in zip(axes, batch["images"], batch["texts"], batch["widths"]):
        ax.imshow(img[0].numpy(), cmap="gray", vmin=-1, vmax=1, aspect="equal")
        ax.axvline(w.item() - 0.5, color="red", lw=0.8)   # red line = where padding starts
        ax.set_title(f"{text!r}  (w={w.item()})", fontsize=8, loc="left")
        ax.set_xticks([]); ax.set_yticks([])   # keep the frame so padding is visible
    fig.tight_layout()
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=120)
    print(f"\nSaved sample grid to {out.resolve()}")


if __name__ == "__main__":
    main()

## Install missing packages
Colab already has PyTorch, OpenCV, NumPy, Pillow, matplotlib, pyarrow and pytest, so this only installs what's missing.
`editdistance` (faster CER/WER) is optional; the code falls back to pure Python without it.

In [ ]:
import importlib.util, subprocess, sys
need = {"torch": "torch", "torchvision": "torchvision", "cv2": "opencv-python", "numpy": "numpy",
        "PIL": "pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "pytest": "pytest",
        "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
        "pytesseract": "pytesseract", "editdistance": "editdistance"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
print("missing:", missing or "nothing")
for pkg in missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    note = " (optional: CER/WER use the pure-Python fallback)" if pkg == "editdistance" else ""
    print(f"  {pkg}: {'installed' if r.returncode == 0 else 'install failed' + note}")

## Unpack archive.zip (about 1.2 GB)
Copied to Colab's disk and unzipped there, which takes a few minutes. Re-running in the same session skips it.

In [ ]:
import os, shutil, time
marker = os.path.join(RAW_DIR, ".unzipped")
if os.path.exists(marker):
    print("Already unzipped this session:", RAW_DIR)
else:
    t = time.time()
    shutil.copy(ARCHIVE, "/content/archive.zip")
    print(f"copied in {time.time()-t:.0f}s"); t = time.time()
    !unzip -q -o /content/archive.zip -d {RAW_DIR}
    os.remove("/content/archive.zip")
    open(marker, "w").close()
    print(f"unzipped in {time.time()-t:.0f}s")
!ls {RAW_DIR}

## Find words.txt and the images
`setup_iam.py` finds the word images and a `words.txt` in any download layout (official tgz, Kaggle `iam_words/`, or a flat folder) and links them into `/content/iam`.

**Which labels get used** (first one that exists wins):
1. `words.txt` in your project folder: the full official file, if you download it from IAM;
2. `words_recovered.txt` in your project folder, made by the label-recovery step below;
3. the `words.txt` inside `archive.zip`. The Kaggle copy is **cut off at e07-072**, so only ~39% of IAM has labels.

In [ ]:
def pick_words_txt():
    for p in (OFFICIAL_WORDS_TXT, RECOVERED_WORDS_TXT):
        if os.path.exists(p):
            return p
    return None

def run_setup():
    wt = pick_words_txt()
    print("labels from:", wt or "the words.txt inside archive.zip")
    extra = f'--words-txt "{wt}"' if wt else ""
    !python scripts/setup_iam.py --raw {RAW_DIR} --out {DATA_ROOT} {extra}

run_setup()

## Recover the missing labels (only needed once)
**Why:** the Kaggle `words.txt` stops part-way through line `e07-072-09-`, but `archive.zip` has the images for all of IAM. Without the missing lines, about 60% of the images can't be used.

**How:** every original IAM PNG has its form's XML stored *inside the file* (a PNG text chunk). That XML gives the word's ID, its text, its tag, its bounding box and its line's segmentation status (ok/err), which are exactly the fields of a `words.txt` line. `recover_labels.py`:
1. first tries the XML inside **your own** images (fast, no download);
2. if they've been stripped, it downloads the Hugging Face copy of IAM ([priyank-m/IAM_words_text_recognition](https://huggingface.co/datasets/priyank-m/IAM_words_text_recognition), ~1.2 GB), whose PNGs still carry the XML, and matches **by word ID**. Images whose ID can't be found fall back to a pixel-hash match.

**Checks it prints, for the report:**
- *rebuilt vs original*: for words that have both, the rebuilt line must equal the original Kaggle line (expect ~100%);
- *pixel check*: the HF image is pixel-identical to yours for a sample of recovered words;
- matched / unmatched / ambiguous counts. Below 90%, nothing is written and training uses the labelled subset.

The result is saved to your project folder as `words_recovered.txt` (original lines first, then a `#--- recovered entries below ---` section), so later sessions and the Step 2 notebook reuse it. Recovered words keep their real IAM IDs, so the **Aachen writer-disjoint splits still apply**.

In [ ]:
if os.path.exists(OFFICIAL_WORDS_TXT):
    print("Using the official full words.txt; no recovery needed.")
elif os.path.exists(RECOVERED_WORDS_TXT):
    print("Reusing", RECOVERED_WORDS_TXT)
else:
    !python scripts/recover_labels.py --data-root {DATA_ROOT} --out "{RECOVERED_WORDS_TXT}" \
        --hf-cache /content/hf_iam --report "{OUTPUT_DIR}/label_recovery.json" \
        --preview "{OUTPUT_DIR}/label_recovery_preview.png"
    if os.path.exists(RECOVERED_WORDS_TXT):
        run_setup()          # re-link with the recovered labels
        from IPython.display import Image as _I, display
        display(_I(os.path.join(OUTPUT_DIR, "label_recovery_preview.png")))

## Run the tests
These cover the charset, CTC decoding, CER/WER, padding, the writer table, label recovery and every download layout. The training tests run in Step 2. Every line should say `passed`.

In [ ]:
!python -m pytest -q tests -k 'not training'

## See the pieces in action

In [ ]:
from src.data.charset import Charset
from src.utils.metrics import cer, wer
cs = Charset()
print("classes (78 chars + blank):", cs.num_classes)
print("encode('Hello'):", cs.encode("Hello"))
h, e, l, o = (cs.encode(c)[0] for c in "helo")
frames = [0, h, h, e, 0, l, l, 0, l, o, o, 0]          # what a CTC model outputs per column
print("greedy CTC on", frames, "->", repr(cs.ctc_greedy_decode(frames)))
print("CER(['hello'], ['helo']) =", cer(["hello"], ["helo"]), "(1 deletion / 5 chars)")
print("WER(['a','cat'], ['a','cot']) =", wer(["a", "cat"], ["a", "cot"]))

## Inspect the real IAM data
Loads the labels, applies the filters, splits by writer, and draws one batch. Opening all images to catch broken files takes a few minutes.

**For the report:**
- **Filtering:** how many words each filter removed (`segmentation_err` = IAM marked the crop as badly segmented, `chars_outside_charset`, `too_long` = over 32 characters, `bad_or_missing_image`, `malformed_line` = a cut-off `words.txt` line). `not_in_any_split` = words from forms that aren't in the Aachen lists.
- **Splits:** words and writers per split. `writers shared between splits` must be 0. With all of IAM, the Aachen splits have 283 / 56 / 161 writers.
- Characters that never appear in train (e.g. `&`, `+`) are just rare in IAM.

In [ ]:
!python scripts/inspect_data.py --data-root {DATA_ROOT} --out "{OUTPUT_DIR}/iam_sample_batch.png" --stats-out "{OUTPUT_DIR}/iam_stats.json"

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(OUTPUT_DIR, "iam_sample_batch.png")))

Each row is one word from a padded batch: all 32 px high, and the red line marks where the real image ends and the padding starts.

**Next:** open **APS360 - HTR Step 2 (Train CRNN)** to train the model. It reuses `words_recovered.txt` automatically.